# 00 - Quickstart: environment and first run

This quickstart notebook runs a compact end-to-end parametric workflow so you can validate the environment and execution path before larger studies.

In practical terms, we will move through preparation, configuration, execution, and verification so every transition in the workflow stays explicit.
As the notebook progresses, each code cell is introduced with context first, then the call itself, and finally quick guidance on how to interpret the result.

Workflow roadmap:
- Preparation: imports, path configuration, and baseline model loading.
- Configuration: output selection, parameter definitions, and problem setup.
- Execution and review: simulation or optimization calls followed by targeted checks.

Main classes, methods, and functions used in this notebook:
- `get_building`
- `accim.utils.reduce_runtime`
- `ParametricSimulation`
- `ParametricSimulation.run_parametric_simulation`
- `WorkspaceArtifactCleaner`
- `ef.get_building`
- `parametric.discover_available_outputs`
- `parametric.select_outputs`
- `parametric.apply_outputs_preflight`
- `parametric.set_output_readers`

Narrative structure used in this notebook:
- Each code cell is preceded by an explanatory markdown block.
- Section headings are contextual to the current notebook and execution block.
- Key arguments are described before they are used in code.
- Execution-oriented cells may include a follow-up markdown note to help interpret results.

## Quickstart run: import workflow dependencies

We start by importing the APIs required for this notebook block.

Keeping imports explicit at the top of each topic notebook makes selective reruns safer when you jump between sections of the series.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

`PROCESSES` is the number of simulations run in parallel: half of the logical CPUs by default.

In [1]:
import os
from pathlib import Path

import accim
import pandas as pd
from besos import eppy_funcs as ef
from accim.utils import WorkspaceArtifactCleaner
from accim.parametric_and_optimisation.main import ParametricSimulation

pd.set_option("display.max_columns", 120)

# Simulations run in parallel: half of the logical CPUs.
PROCESSES = max(1, (os.cpu_count() or 2) // 2)


## Quickstart run: define paths and execution folders

This block defines supporting paths and run constants used by later cells.

Centralizing these values reduces hidden coupling and keeps the execution flow easier to audit.

Variables created in this cell:
- `PROJECT_ROOT`: reused in subsequent workflow blocks.
- `idf_file`: reused in subsequent workflow blocks.
- `epw_list`: reused in subsequent workflow blocks.
- `RESULTS_DIRNAME`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `PROJECT_ROOT`, `idf_file`, `epw_list`, `RESULTS_DIRNAME`.

In [2]:
PROJECT_ROOT = Path.cwd()

idf_file = "OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_high_performance.idf"
epw_list = ["Seville.epw"]
RESULTS_DIRNAME = "nb00_quickstart_results"

print("Project root:", PROJECT_ROOT)
print("IDF exists:", (PROJECT_ROOT / idf_file).exists())
print("EPW exists:", (PROJECT_ROOT / epw_list[0]).exists())
print("Note: quickstart uses the high-performance envelope IDF only.")


Project root: .
IDF exists: True
EPW exists: True
Note: quickstart uses the high-performance envelope IDF only.


## Quickstart run: capture baseline workspace state for cleanup

We initialize artifact tracking so generated files can be reviewed and cleaned in a controlled way.

This is especially useful in iterative notebook sessions where repeated runs can quickly accumulate temporary folders and CSV files.

Key arguments used in this cell and why they matter:
- `workspace_root`: defines this setting for the current call.

Variables created in this cell:
- `artifact_cleaner`: reused in subsequent workflow blocks.
- `baseline_files`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `artifact_cleaner`, `baseline_files`.

In [3]:
artifact_cleaner = WorkspaceArtifactCleaner(workspace_root=PROJECT_ROOT)
baseline_files = artifact_cleaner.capture_initial_state()
print("Baseline file count:", len(baseline_files))


Baseline file count: 7


## Quickstart run: load the baseline IDF model

Load the base building model used by subsequent runs.

This is where the baseline IDF is loaded for all downstream operations. Later cells reuse this object for output setup, parameterization, and execution.

Variables created in this cell:
- `building`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `building`.

In [4]:
building = ef.get_building(idf_file)


## Quickstart run: apply runtime-reduction settings

Apply runtime reductions to keep notebook runs faster during exploration.

This block shortens exploratory runs by reducing simulation burden. Use it for prototyping and switch to fuller settings when producing final results.

`reduce_runtime` also changes the run period, to 1 January by default, so its run period arguments keep the full year: the results stay annual. `Run_Simulation_for_Sizing_Periods = "No"` keeps the design days of the sample model out of the annual results (they are still used to size the HVAC system).

Key arguments used in this cell and why they matter:
- `idf_object`: defines this setting for the current call.
- `timesteps`: defines this setting for the current call.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [5]:

# Keep quickstart fast. Increase timesteps for production studies.
accim.utils.reduce_runtime(
    idf_object=building,
    timesteps=2,
    runperiod_begin_month=1,
    runperiod_begin_day_of_month=1,
    runperiod_end_month=12,
    runperiod_end_day_of_month=31,
)
building.idfobjects["SimulationControl"][0].Run_Simulation_for_Sizing_Periods = "No"


Solar distribution has been set to MinimalShadowing.
Shading Calculation Update Frequency was previously set to 20 days, and it has been modified to 20 days.
Maximum Figures in Shadow Overlap Calculations was previously set to 15000, and it has been modified to 200.
Number of Timesteps per Hour was previously set to 6, and it has been modified to 2.


## Quickstart run: initialize parametric simulation object

Create the parametric simulation object with weather context and model settings.

This creates the parametric simulation object that will orchestrate outputs, parameters, and run plans. From here on, the workflow uses this object as the main controller.

Key arguments used in this cell and why they matter:
- `building`: loaded IDF model used in this workflow.
- `parameters_type`: parameter family used in the simulation setup.
- `epws`: list of EPW weather files used to run scenarios.
- `output_frequencies`: time frequency requested for outputs.

Variables created in this cell:
- `parametric`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `parametric`.

In [6]:

parametric = ParametricSimulation(
	building=building,
	parameters_type="accim custom model",
	epws=epw_list,
	output_frequencies=["hourly"],
)



--------------------------------------------------------
Adaptive-Comfort-Control-Implemented Model (ACCIM) v1.0.0
--------------------------------------------------------

This tool allows to apply adaptive setpoint temperatures. 
For further information, please read the documentation: 
https://accim.readthedocs.io/en/master/
For a visual understanding of the tool, please visit the following jupyter notebooks:
-    Using AddAccis() to apply adaptive setpoint temperatures
https://accim.readthedocs.io/en/master/jupyter_notebooks/addAccis/using_addAccis.html
-    Using rename_epw_files() to rename the EPWs for proper data analysis after simulation
https://accim.readthedocs.io/en/master/jupyter_notebooks/rename_epw_files/using_rename_epw_files.html
-    Using run_ep() to directly run simulations with EnergyPlus
https://accim.readthedocs.io/en/master/jupyter_notebooks/runEp/using_runEp.html
-    Using the class Table() for data analysis
https://accim.readthedocs.io/en/master/jupyter_noteb

Relative humidity setpoint schedule: Always 50.00 Schedule has been added
Heating Fanger comfort setpoint: Always -0.5 Schedule has been added
Cooling Fanger comfort setpoint: Always  0.1 Schedule has been added


Zone CO2 setpoint: Always 900ppm Schedule has been added
Min CO2 concentration: Always 600ppm Schedule has been added
Generic contaminant setpoint: Always 0.5ppm Schedule has been added
Air distribution effectiveness (always 1) Schedule has been added


VRF Heating Cooling (Northern Hemisphere) Schedule has been added
DefaultFanEffRatioCurve Curve:Cubic Object has been added
VRFTUCoolCapFT Curve:Cubic Object has been added
VRFTUHeatCapFT Curve:Cubic Object has been added
VRFCoolCapFTBoundary Curve:Cubic Object has been added
VRFCoolEIRFTBoundary Curve:Cubic Object has been added
CoolingEIRLowPLR Curve:Cubic Object has been added
VRFHeatCapFTBoundary Curve:Cubic Object has been added
VRFHeatEIRFTBoundary Curve:Cubic Object has been added
HeatingEIRLowPLR Curve:Cubic Object has been added
DefaultFanPowerRatioCurve Curve:Exponent Object has been added
DXHtgCoilDefrostEIRFT Curve:Biquadratic Object has been added
VRFCoolCapFT Curve:Biquadratic Object has been added
VRFCoolCapFTHi Curve:Biquadratic Object has been added
VRFCoolEIRFT Curve:Biquadratic Object has been added


VRFCoolEIRFTHi Curve:Biquadratic Object has been added
VRFHeatCapFT Curve:Biquadratic Object has been added
VRFHeatCapFTHi Curve:Biquadratic Object has been added
VRFHeatEIRFT Curve:Biquadratic Object has been added
VRFHeatEIRFTHi Curve:Biquadratic Object has been added
CoolingLengthCorrectionFactor Curve:Biquadratic Object has been added
VRF Piping Correction Factor for Length in Heating Mode Curve:Biquadratic Object has been added
VRF Heat Recovery Cooling Capacity Modifier Curve:Biquadratic Object has been added
VRF Heat Recovery Cooling Energy Modifier Curve:Biquadratic Object has been added
VRF Heat Recovery Heating Capacity Modifier Curve:Biquadratic Object has been added
VRF Heat Recovery Heating Energy Modifier Curve:Biquadratic Object has been added
VRFACCoolCapFFF Curve:Quadratic Object has been added
CoolingEIRHiPLR Curve:Quadratic Object has been added
VRFCPLFFPLR Curve:Quadratic Object has been added
HeatingEIRHiPLR Curve:Quadratic Object has been added
CoolingCombRatio Cu

AHST_Sch_Floor_2_Residential_Living_Occupants Schedule has been added
ACST_Sch_Floor_2_Residential_Living_Occupants Schedule has been added
Added - SetComfTemp Program
Added - CountHours_Floor_1_Residential_Living_Occupants Program
Added - CountHours_Floor_2_Residential_Living_Occupants Program
Added - SetAppLimits Program
Added - ApplyCAT Program
Added - SetAST_Master Program
Added - SetAST_CS0_0 Program
Added - SetAST_CS1_0 Program
Added - SetAST_CS1_1 Program
Added - SetAST_CS2_0 Program
Added - SetAST_CS2_1 Program
Added - SetAST_CS3_0 Program
Added - SetAST_CS3_1 Program
Added - SetAST_CS4_0 Program
Added - SetAST_CS5_0 Program
Added - SetAST_CS6_0 Program
Added - SetAST_CS6_1 Program


Added - SetAST_CS7_0 Program
Added - SetAST_CS7_1 Program
Added - SetAST_CS8_0 Program
Added - SetAST_CS8_1 Program
Added - SetAST_CS9_0 Program
Added - SetAST_CS9_1 Program
Added - SetAST_CS10_0 Program
Added - SetAST_CS10_1 Program
Added - SetAST_CS11_0 Program
Added - SetAST_CS11_1 Program
Added - SetAST_CS12_0 Program
Added - SetAST_CS12_1 Program
Added - SetAST_CS13_0 Program
Added - SetAST_CS13_1 Program
Added - SetAST_CS13_2 Program
Added - SetAST_CS13_3 Program
Added - SetAST_CS14_0 Program
Added - SetAST_CS14_1 Program
Added - SetAST_CS14_2 Program
Added - SetAST_CS14_3 Program
Added - SetAST_CS15_0 Program
Added - SetAST_CS15_1 Program
Added - SetAST_CS16_0 Program
Added - SetAST_CS16_1 Program
Added - SetAST_CS17_0 Program
Added - SetAST_CS17_1 Program
Added - SetAST_CS18_0 Program
Added - SetAST_CS18_1 Program
Added - SetAST_CS19_0 Program
Added - SetAST_CS19_1 Program
Added - SetAST_CS20_0 Program
Added - SetAST_CS20_1 Program
Added - SetAST_CS21_0 Program
Added - SetAST_C

Added - CountHoursNoApp_Floor_2_Residential_Living_Occupants Program
Added - SetGeoVarFloor_2_Residential_Living_Occupants Program
Added - SetInputData_Floor_1_Residential_Living_Occupants Program
Added - SetInputData_Floor_2_Residential_Living_Occupants Program
Added - SetVST Program
Added - ApplyAST_Floor_1_Residential_Living_Occupants Program
Added - ApplyAST_Floor_2_Residential_Living_Occupants Program
Added - SetMyVOF_Floor_1_Zone_Win Program
Added - SetWindowOperation_Floor_1_Zone_Win Program
Added - SetMyVOF_Floor_2_Zone_Win Program
Added - SetWindowOperation_Floor_2_Zone_Win Program
Added - RunACCIS_Floor_1_Residential_Living_Occupants Program
Added - RunACCIS_Floor_2_Residential_Living_Occupants Program
Added - Comfort Temperature Output Variable
Added - Adaptive Cooling Setpoint Temperature Output Variable
Added - Adaptive Heating Setpoint Temperature Output Variable
Added - Adaptive Cooling Setpoint Temperature_No Tolerance Output Variable
Added - Adaptive Heating Setpoint T

Added - Environment Site Outdoor Air Drybulb Temperature Reporting Frequency Hourly Output:Variable data
Added - Environment Site Wind Speed Reporting Frequency Hourly Output:Variable data
Added - Environment Site Outdoor Air Relative Humidity Reporting Frequency Hourly Output:Variable data
Added - AHST_Sch_Floor_1_Residential_Living_Occupants Schedule Value Reporting Frequency Hourly Output:Variable data
Added - ACST_Sch_Floor_1_Residential_Living_Occupants Schedule Value Reporting Frequency Hourly Output:Variable data
Added - AHST_Sch_Floor_2_Residential_Living_Occupants Schedule Value Reporting Frequency Hourly Output:Variable data
Added - ACST_Sch_Floor_2_Residential_Living_Occupants Schedule Value Reporting Frequency Hourly Output:Variable data
Added - Vent_Sch_Floor_1_Zone Schedule Value Reporting Frequency Hourly Output:Variable data
Added - Vent_Sch_Floor_2_Zone Schedule Value Reporting Frequency Hourly Output:Variable data
Added - * VRF Heat Pump Cooling Electricity Energy Rep

## Quickstart run: discover available meter and variable outputs

Inspect available meters and variables before selecting valid outputs.

Before selecting outputs, we inspect what meters and variables are currently available. That discovery step reduces mistakes when requesting objective or analysis outputs.

Key arguments used in this cell and why they matter:
- `reduce_sim_time`: enables a fast scan mode when discovering outputs.
- `prefer`: preferred backend for output discovery.

Variables created in this cell:
- `df_meters_available`: reused in subsequent workflow blocks.
- `df_vars_available`: reused in subsequent workflow blocks.
- `outputs_meta`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Check that the discovered meter/variable tables are not unexpectedly empty.
- Confirm the reported discovery source matches your preferred backend.

In [7]:
available_outputs = parametric.discover_available_outputs(
	reduce_sim_time=True,  # lightweight discovery helper run
	prefer="testsimeplus",  # fallback policy when multiple discovery backends exist
)
df_meters_available = available_outputs["meters"]
df_vars_available = available_outputs["variables"]
outputs_meta = available_outputs["meta"]

print("Discovery source:", outputs_meta.get("source"))
print("Meters available:", len(df_meters_available))
print("Variables available:", len(df_vars_available))


Solar distribution is already set to MinimalShadowing, therefore no action has been performed.
Shading Calculation Update Frequency was previously set to 20 days, and it has been modified to 20 days.
Maximum Figures in Shadow Overlap Calculations was previously set to 200, and it has been modified to 200.
Number of Timesteps per Hour was previously set to 2, and it has been modified to 2.


Discovery source: testsimeplus
Meters available: 6
Variables available: 160


## Quickstart run: select objective and analysis outputs

Select target outputs and validate them before applying changes to the model.

This section applies explicit output selection and preflight validation. Validating before execution keeps result schemas stable across runs.

Key arguments used in this cell and why they matter:
- `meters`: energy meters to include as outputs.
- `match`: matching rule used to select outputs.
- `on_missing`: behavior when a requested output is missing.
- `suggest`: shows suggestions when no exact match is found.

Variables created in this cell:
- `objective_meters`: reused in subsequent workflow blocks.
- `df_meters_sel`: reused in subsequent workflow blocks.
- `_`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm the selected tables include all objective outputs you plan to track.
- Verify preflight did not report blocking missing outputs.

In [8]:
objective_meters = ["Heating:Electricity", "Cooling:Electricity"]
selection = parametric.select_outputs(
	meters=objective_meters,  # exact output goals for this quickstart
	match="case_insensitive",  # robust matching for common casing differences
	on_missing="raise",  # fail early if an objective output cannot be found
	suggest=True,  # print close matches when selection fails
)
df_meters_sel = selection["meters"]


## Quickstart run: run output preflight checks before execution

Select target outputs and validate them before applying changes to the model.

This section applies explicit output selection and preflight validation. Validating before execution keeps result schemas stable across runs.

Key arguments used in this cell and why they matter:
- `df_meters_sel`: table of selected meters to apply to the IDF.
- `df_vars_sel`: table of selected variables to apply to the IDF.
- `clean_mode`: cleanup mode applied before adding outputs.
- `validate_before_apply`: whether to validate before editing the IDF.
- `validate_after_apply`: whether to validate after editing the IDF.
- `on_missing`: behavior when a requested output is missing.

Variables created in this cell:
- `preflight`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm the selected tables include all objective outputs you plan to track.
- Verify preflight did not report blocking missing outputs.

In [9]:
preflight = parametric.apply_outputs_preflight(
	df_meters_sel=df_meters_sel,
	df_vars_sel=None,
	clean_mode="all",  # remove legacy output objects and keep selected ones
	validate_before_apply=False,
	validate_after_apply=True,  # confirm selected outputs exist after apply
	on_missing="raise",
)

print("Missing meters after apply:", len(preflight["verification"]["meters"]["missing_in_idf"]))


Missing meters after apply: 0


## Quickstart run: apply selected output schema to simulation settings

Apply the validated output meter/variable schema to the simulation object.

This block applies the validated output-meter and output-variable schema to the active simulation object. From this point on, execution calls will use this explicit output definition.

Key arguments used in this cell and why they matter:
- `df_output_meter`: final meter table used by the simulation.

What to verify right after running this cell:
- Confirm output schema assignment completed without warnings.
- Verify target meter/variable names match the intended analysis outputs.

In [10]:
parametric.set_output_readers(df_output_meter=df_meters_sel)


## Quickstart run: define decision variables and objective directions

Set parameter ranges or values and build the study problem.

Here we define the decision space and objective orientation used in simulation or optimization. These choices directly determine how the final result space will look.

Key arguments used in this cell and why they matter:
- `accis_params_dict`: ACCIS/aPMV parameter definition used in the study.
- `num_samples`: number of samples in the sampling plan.

Variables created in this cell:
- `accis_parameters`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `accis_parameters`.

In [11]:
# Define a tiny parameter space and attach the EPProblem.
# Each tuple is interpreted as a numeric range: (min, max).
accis_parameters = {
	"custom_slope": (0.2, 0.8),
	"custom_intercept": (10, 22),
	"custom_offset": (2.0, 5.0),
}

parametric.set_parameters(accis_params_dict=accis_parameters)
parametric.set_problem()  # builds EPProblem from selected outputs + parameter definition
parametric.sampling_lhs(num_samples=4)  # small Latin Hypercube sample for demonstration

print("Sample size:", len(parametric.parameters_values_df))
parametric.parameters_values_df.head()


The following parameters are not included in the parameters to be set, and have not been defined yet (i.e. the value is 0): ['custom_cooling_upper_limit', 'custom_cooling_lower_limit', 'custom_heating_upper_limit', 'custom_heating_lower_limit']
Sample size: 4


,custom_slope,custom_intercept,custom_offset,epw
0,0.295693,20.378743,4.146409,Seville.epw
1,0.453387,15.335966,2.566228,Seville.epw
2,0.690929,10.690789,3.476806,Seville.epw
3,0.529724,16.751711,4.453072,Seville.epw


## Quickstart run: execute parametric run and inspect output rows

This is the execution checkpoint for the configured parametric plan.

After the run, verify row counts and core objective columns immediately so issues are caught before post-processing.

Key arguments used in this cell and why they matter:
- `out_dir`: output folder for generated artifacts and results.
- `df`: case plan dataframe executed in this run.
- `processes`: number of parallel processes for simulation runs.
- `keep_input`: whether generated input files are preserved.
- `keep_dirs`: whether per-run directories are preserved.

Variables created in this cell:
- `outputs`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Compare returned row count with the expected number of plan combinations.
- Inspect the first rows to ensure objective columns are present and populated.

In [12]:
outputs = parametric.run_parametric_simulation(
	out_dir=RESULTS_DIRNAME,  # output folder for batch artifacts and logs
	df=parametric.parameters_values_df,  # explicit sample plan to execute
	processes=PROCESSES,  # increase with care; parallel overhead can dominate tiny runs
	keep_input=True,  # keep generated input models for auditability
	keep_dirs=False,  # remove per-run directories to reduce disk usage
)
print("Rows:", len(outputs))
print(outputs.head())


  [info] simulation_summary updated for parametric after run_parametric_simulation.
Rows: 4
   Heating:Electricity  Cooling:Electricity  custom_slope  custom_intercept  \
0         8.079793e+07         3.491117e+10      0.295693         20.378743   
1         1.411839e+08         5.236008e+10      0.453387         15.335966   
2         1.414289e+08         4.537385e+10      0.690929         10.690789   
3         1.137903e+08         2.880814e+10      0.529724         16.751711   

   custom_offset      epw                                                idf  
0       4.146409  Seville  OSM_TestResidentialUnit_v01_onlygeometry_SchNa...  
1       2.566228  Seville  OSM_TestResidentialUnit_v01_onlygeometry_SchNa...  
2       3.476806  Seville  OSM_TestResidentialUnit_v01_onlygeometry_SchNa...  
3       4.453072  Seville  OSM_TestResidentialUnit_v01_onlygeometry_SchNa...  


### Reading the result

Review the row count and confirm it matches the expected combinations from your plan and EPW set.

## Quickstart run: preview execution outputs and row counts

Inspect a compact preview of execution outputs before proceeding.

This cell previews key output rows or summary counts after execution. Use it to quickly validate shape and columns before moving to deeper analysis.

What to verify right after running this cell:
- Confirm row counts and first records look consistent with the execution settings.
- Check that expected objective/output columns are present before plotting or MCDM steps.

In [13]:
outputs


,Heating:Electricity,Cooling:Electricity,custom_slope,custom_intercept,custom_offset,epw,idf
0,8.079793e+07,3.491117e+10,0.295693,20.378743,4.146409,Seville,OSM_TestResidentialUnit_v01_onlygeometry_SchNa...
1,1.411839e+08,5.236008e+10,0.453387,15.335966,2.566228,Seville,OSM_TestResidentialUnit_v01_onlygeometry_SchNa...
2,1.414289e+08,4.537385e+10,0.690929,10.690789,3.476806,Seville,OSM_TestResidentialUnit_v01_onlygeometry_SchNa...
3,1.137903e+08,2.880814e+10,0.529724,16.751711,4.453072,Seville,OSM_TestResidentialUnit_v01_onlygeometry_SchNa...


## Quickstart run: inspect generated artifacts before cleanup

This section audits generated artifacts and applies the configured cleanup policy.

Keeping cleanup explicit avoids accidental deletion while still preventing accumulation of stale run artifacts.

Key arguments used in this cell and why they matter:
- `max_items`: maximum number of items shown in logs.
- `allow_patterns`: file patterns that are allowed for cleanup.
- `deny_patterns`: file patterns protected from cleanup.
- `dry_run`: whether cleanup only reports actions without deleting.
- `remove_empty_dirs`: whether empty directories are removed after cleanup.

Variables created in this cell:
- `generated_files`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `generated_files`.

In [14]:
generated_files = artifact_cleaner.print_generated_files(max_items=200)

# Force-remove every generated artifact inside the dedicated result folder,
# including generated IDF files.
artifact_cleaner.delete_generated_files(
	allow_patterns=[f"{RESULTS_DIRNAME}/**"],
	deny_patterns=[],
	dry_run=True,
	remove_empty_dirs=True,
)


Generated files detected: 6
 - nb00_quickstart_results/accim_idf_backup_OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_high_performance_pre_parametric_20261102_091236.idf
 - nb00_quickstart_results/outputs_param_simulation_20261102_091326.csv
 - nb00_quickstart_results/outputs_param_simulation_20261102_091326.json
 - nb00_quickstart_results/outputs_param_simulation_20261102_091326.pkl
 - nb00_quickstart_results/outputs_param_simulation_20261102_091326.xlsx
 - nb00_quickstart_results/outputs_param_simulation_batches/outputs_param_simulation_batch_00001_20261102_091326_506492.pkl
Files planned for deletion: 6
 - nb00_quickstart_results/accim_idf_backup_OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_high_performance_pre_parametric_20261102_091236.idf
 - nb00_quickstart_results/outputs_param_simulation_20261102_091326.csv
 - nb00_quickstart_results/outputs_param_simulation_20261102_091326.json
 - nb00_quickstart_results/outputs_param_simulation_20261102_091326.pkl
 - 

['nb00_quickstart_results/accim_idf_backup_OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_high_performance_pre_parametric_20261102_091236.idf',
 'nb00_quickstart_results/outputs_param_simulation_20261102_091326.csv',
 'nb00_quickstart_results/outputs_param_simulation_20261102_091326.json',
 'nb00_quickstart_results/outputs_param_simulation_20261102_091326.pkl',
 'nb00_quickstart_results/outputs_param_simulation_20261102_091326.xlsx',
 'nb00_quickstart_results/outputs_param_simulation_batches/outputs_param_simulation_batch_00001_20261102_091326_506492.pkl']

## Quickstart run: remove generated artifacts from this run

This section audits generated artifacts and applies the configured cleanup policy.

Keeping cleanup explicit avoids accidental deletion while still preventing accumulation of stale run artifacts.

`DELETE_RESULTS` is False by default, so nothing is deleted. Set it to True to delete the files listed above.

Key arguments used in this cell and why they matter:
- `allow_patterns`: file patterns that are allowed for cleanup.
- `deny_patterns`: file patterns protected from cleanup.
- `dry_run`: whether cleanup only reports actions without deleting.
- `remove_empty_dirs`: whether empty directories are removed after cleanup.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [15]:
DELETE_RESULTS = False

if DELETE_RESULTS:
    artifact_cleaner.delete_generated_files(
        allow_patterns=[f"{RESULTS_DIRNAME}/**"],
        deny_patterns=[],
        dry_run=False,
        remove_empty_dirs=True,
    )
